# Imbalance-Targeted Augmentation (Pillar 5b)

Implements Sub-pillar 5b from the paper (§2.5, Figure 11): **inverse-frequency
oversampling of underrepresented organ / categorization / regional-anatomy classes**,
with Shannon entropy and the Gini coefficient computed **before and after** to show
the correction.

**Magnification** is balanced alongside the paper's three fields. It is not named in
§3.5, but it is imbalanced on the same order (10.4x ratio, Gini 0.369) and it is the one
field whose imbalance has a demonstrated downstream cost - the Benchmark 4 analysis below
finds augmented 400x images rated lowest by every rater. Unlike the other three it is a
property of the *image* rather than the *case*, so balancing it leaves case-level
representation untouched.

## Selection, not generation

Every one of the 229 usable images already carries **30 augmentations on disk**
(`pathopen_data/processed/data.json` -> `Images[].Augmented`). So a class-balanced
variant does not require running the transform pipeline again - it is a question of
*how many of each image's existing 30 augmentations to keep*. Rare classes keep more,
common classes keep fewer.

This matters for the paper's claim: the balanced variant costs no new image synthesis
and introduces no new clinical information, which is exactly the caveat §2.5's
reviewer-concerns section asks to be stated explicitly.

## Why the uniform 30x augmentation cannot be the "after" distribution

The existing augmentation multiplies **every** image by the same factor of 30. Scaling
every class by a constant leaves the class *proportions* untouched, so entropy and Gini
are mathematically **unchanged** - identical to 3 decimal places, not merely similar.
Reporting that as Figure 11 Panel B would show a null result and prove nothing. The
targeted variant built here is what makes the before/after comparison meaningful.

## Method

For a class $c$ with raw count $n_c$, the inverse-frequency multiplier is

$$m_c = \mathrm{clip}\!\left(\frac{n_{\max}}{n_c},\; 1,\; M\right)$$

where $M$ (`MAX_MULTIPLIER`) caps how far a rare class may be oversampled. The cap is
not cosmetic: without it a singleton class would be replicated 49x, and §2.5's stated
reviewer concern is precisely *"overfitting risk from repeated oversampling of small
classes should be acknowledged, with a maximum oversampling multiplier considered."*
The multiplier is also bounded above by the 30 augmentations that actually exist per
image - we can never keep more than were generated.

Balancing targets one metadata field at a time (they are not independent - organ and
regional anatomy are nested - so a single joint balancing would be ill-posed).

In [ ]:
import collections
import json
import os

import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED_DATA_JSON = os.path.join(REPO_ROOT, "pathopen_data", "processed", "data.json")

# Two destinations, split by what the artifact IS:
#   OUTPUT_DIR    - analysis tables (entropy/Gini, sweeps, magnification quality). These
#                   are results about the data, and live with the other analysis outputs.
#   MANIFEST_DIR  - the balanced-variant manifests. These ARE dataset artifacts: they
#                   define which augmentation files a training run consumes, so they
#                   belong under pathopen_data/ with everything else PathOPEN ships,
#                   not in an analysis folder someone would have to know to look in.
OUTPUT_DIR = os.path.join(os.getcwd(), "augmentation_balance_output")
MANIFEST_DIR = os.path.join(REPO_ROOT, "pathopen_data", "augmented", "balanced_manifests")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

# The three fields §3.5 names for Pillar 5b, plus Magnification.
#
# Magnification is not in the paper's list, but it is imbalanced on the same order as the
# others (10.4x ratio, Gini 0.369 - between Regional Anatomy's 0.320 and Organ's 0.450),
# and it is the one field where imbalance has a *known* downstream effect: the Benchmark 4
# analysis below shows augmented 400x images scoring lowest with every rater. A model
# trained on the uniform augmentation sees 200x/100x images ~10x more often than 50x ones.
#
# It also differs structurally from the other three: magnification is a property of the
# IMAGE, while organ/categorization/regional-anatomy are properties of the CASE. That
# makes it the only field here that can be balanced without touching case-level
# representation, so it composes with the others rather than competing with them.
METADATA_FIELDS = ["Organ", "Categorization", "Regional Anatomy", "Magnification"]

# Cap on inverse-frequency oversampling. 30 is the hard ceiling (only 30 augmentations
# exist per image); 10 is the reported default - it closes most of the imbalance while
# keeping any single rare case from dominating its class. Both are swept below so the
# choice is shown to be a trade-off rather than asserted.
MAX_MULTIPLIER = 10
AVAILABLE_AUGMENTATIONS = 30

PROCESSED_DATA_JSON, OUTPUT_DIR, MANIFEST_DIR


## Load cases and flatten to the image level

Metadata (organ, categorization, regional anatomy) lives on the **case**, but
augmentation happens per **image**, and a case may hold up to 3 images. The image is
therefore the unit of analysis, with each image inheriting its case's metadata.

Two image entries (cases 9 and 79) are malformed - empty `Directory`, blank
`Magnification`, and zero augmentations. They are excluded and reported, not silently
dropped, since they would otherwise contribute a phantom class of unaugmentable images.

In [ ]:
with open(PROCESSED_DATA_JSON) as f:
    cases = json.load(f)

records, skipped = [], []
for case in cases:
    for image in case["Images"]:
        n_aug = len(image.get("Augmented", []))
        if n_aug == 0 or not image.get("Directory", "").strip():
            skipped.append({"case_id": case["Case ID"], "n_augmented": n_aug})
            continue
        records.append({
            "case_id": case["Case ID"],
            "image_id": os.path.basename(image["Directory"]),
            "Organ": str(case.get("Organ", "")).strip(),
            "Categorization": str(case.get("Categorization", "")).strip(),
            "Regional Anatomy": str(case.get("Regional Anatomy", "")).strip(),
            "Magnification": image.get("Magnification", "").strip(),
            "n_augmented_available": n_aug,
        })

images_df = pd.DataFrame(records)
print(f"cases={len(cases)}  usable images={len(images_df)}  skipped={len(skipped)}")
if skipped:
    print("  excluded (no augmentations / empty directory):", skipped)
print(f"augmentations per image: min={images_df.n_augmented_available.min()} "
      f"max={images_df.n_augmented_available.max()}")
images_df.head()

## Entropy and Gini

**Shannon entropy** $H = -\sum p_c \log p_c$ measures how evenly spread the classes are;
it is reported both raw and normalized by $H_{\max} = \log(k)$ so distributions with
different class counts stay comparable. Normalized entropy of 1.0 is perfectly uniform.

**Gini coefficient** measures concentration on the same counts: 0 is perfectly equal,
and it rises toward 1 as a few classes absorb more of the mass. The two are reported
together because they disagree usefully - entropy is dominated by the many small
classes, Gini by the few large ones, and an intervention can move one much more than
the other.

In [ ]:
def shannon_entropy(counts) -> tuple:
    """(H, H_max, H/H_max) in nats. Zero/negative counts are dropped - a class with no
    members contributes nothing to entropy and would make log(0) undefined."""
    counts = np.asarray([c for c in counts if c > 0], dtype=float)
    if len(counts) < 2:
        return 0.0, 0.0, np.nan
    p = counts / counts.sum()
    h = float(-(p * np.log(p)).sum())
    h_max = float(np.log(len(counts)))
    return h, h_max, h / h_max if h_max > 0 else np.nan


def gini(counts) -> float:
    """Gini coefficient over class counts (0 = perfectly balanced).

    Uses the sorted-rank form: G = 2*sum(i*n_i)/(k*sum(n)) - (k+1)/k. Counts must be
    sorted ascending for the rank weights to mean anything."""
    counts = np.asarray([c for c in counts if c > 0], dtype=float)
    if len(counts) < 2:
        return 0.0
    counts = np.sort(counts)
    k = len(counts)
    index = np.arange(1, k + 1)
    return float((2 * (index * counts).sum()) / (k * counts.sum()) - (k + 1) / k)


def distribution_stats(counts, label: str, field: str) -> dict:
    counts = [c for c in counts if c > 0]
    h, h_max, h_norm = shannon_entropy(counts)
    return {
        "field": field, "stage": label, "n_classes": len(counts), "n_items": int(sum(counts)),
        "min_class": int(min(counts)), "max_class": int(max(counts)),
        "imbalance_ratio": round(max(counts) / min(counts), 1),
        "shannon_entropy": round(h, 4), "entropy_max": round(h_max, 4),
        "entropy_normalized": round(h_norm, 4), "gini": round(gini(counts), 4),
    }


# Sanity check on known-answer inputs before trusting these on real data.
assert abs(gini([5, 5, 5, 5])) < 1e-9, "Gini of a uniform distribution must be 0"
assert abs(shannon_entropy([1, 1, 1, 1])[2] - 1.0) < 1e-9, "uniform normalized entropy must be 1"
assert gini([1, 1, 1, 97]) > 0.5, "Gini must be high for a concentrated distribution"
print("entropy/gini self-checks passed")

## Figure 11 Panel A: the raw imbalance

Baseline entropy and Gini per metadata field, computed on raw case-level counts.

In [ ]:
baseline_rows = []
raw_counts = {}
for field in METADATA_FIELDS:
    counts = collections.Counter(images_df[field].replace("", "(unspecified)"))
    raw_counts[field] = counts
    baseline_rows.append(distribution_stats(counts.values(), "raw", field))

baseline_df = pd.DataFrame(baseline_rows)
print(baseline_df.to_string(index=False))
baseline_df

## Build the targeted (inverse-frequency) variant

`targeted_multipliers` returns, per image, how many of its 30 existing augmentations to
keep so the chosen field's class distribution flattens. The multiplier is capped twice:
by `MAX_MULTIPLIER` (the overfitting guard) and by the augmentations that physically
exist (30).

In [ ]:
def targeted_multipliers(df: pd.DataFrame, field: str, max_multiplier: int) -> pd.Series:
    """Per-image count of augmentations to retain, inverse to its class's frequency.

    A class holding n_c images gets multiplier clip(n_max / n_c, 1, max_multiplier),
    further clipped to the augmentations actually available for that image. The result
    is >= 1 everywhere: no image is ever dropped, only differentially oversampled."""
    class_counts = df[field].replace("", "(unspecified)").value_counts()
    n_max = class_counts.max()
    raw_mult = df[field].replace("", "(unspecified)").map(lambda c: n_max / class_counts[c])
    capped = raw_mult.clip(lower=1, upper=max_multiplier).round().astype(int)
    return np.minimum(capped, df["n_augmented_available"])


def targeted_class_counts(df: pd.DataFrame, field: str, multipliers: pd.Series) -> collections.Counter:
    """Post-augmentation count per class = sum of retained augmentations per image."""
    counts = collections.Counter()
    labels = df[field].replace("", "(unspecified)")
    for label, mult in zip(labels, multipliers):
        counts[label] += int(mult)
    return counts


def uniform_class_counts(df: pd.DataFrame, field: str, factor: int) -> collections.Counter:
    """The EXISTING uniform 30x augmentation, for contrast."""
    counts = collections.Counter()
    labels = df[field].replace("", "(unspecified)")
    for label in labels:
        counts[label] += factor
    return counts


stage_rows, targeted_counts = [], {}
for field in METADATA_FIELDS:
    stage_rows.append(distribution_stats(raw_counts[field].values(), "raw", field))
    uni = uniform_class_counts(images_df, field, AVAILABLE_AUGMENTATIONS)
    stage_rows.append(distribution_stats(uni.values(), "uniform_30x", field))
    mult = targeted_multipliers(images_df, field, MAX_MULTIPLIER)
    tgt = targeted_class_counts(images_df, field, mult)
    targeted_counts[field] = tgt
    stage_rows.append(distribution_stats(tgt.values(), f"targeted_max{MAX_MULTIPLIER}x", field))

balance_df = pd.DataFrame(stage_rows)
print(balance_df.to_string(index=False))
balance_df

## Sensitivity to the multiplier cap

`MAX_MULTIPLIER` is the one free parameter, and it trades imbalance correction against
overfitting risk. Sweeping it shows how much of the correction each cap buys, so the
reported choice can be justified rather than asserted.

In [ ]:
sweep_rows = []
for cap in [1, 2, 3, 5, 10, 15, 20, 30]:
    for field in METADATA_FIELDS:
        mult = targeted_multipliers(images_df, field, cap)
        counts = targeted_class_counts(images_df, field, mult)
        stats = distribution_stats(counts.values(), f"max{cap}x", field)
        stats["max_multiplier"] = cap
        stats["total_augmented_images"] = int(mult.sum())
        sweep_rows.append(stats)

sweep_df = pd.DataFrame(sweep_rows)
pivot = sweep_df.pivot(index="max_multiplier", columns="field",
                       values=["entropy_normalized", "gini"])
print(pivot.to_string())
print("\ntotal augmented images retained, by cap:")
print(sweep_df[sweep_df.field == "Organ"][["max_multiplier", "total_augmented_images"]].to_string(index=False))
sweep_df

## Magnification-stratified augmentation quality

Replaces the original Figure 10 Panel B ("breakdown by transform type"), which is not
computable: the 5 transforms per augmentation were sampled at random and never recorded,
so there is no per-image transform manifest to group by.

Magnification is a defensible substitute and arguably a more interesting question: it is
recorded per image, it is one of PathOPEN's stated structural advantages (up to 5
magnification levels per case, Table 3), and "does augmentation degrade high-magnification
images more than low?" is exactly what a reviewer would ask of a morphology-preserving
transform pipeline. Fine nuclear detail at 400x is more fragile than tissue architecture
at 20x, so a magnification-dependent drop would be a real finding.

This joins the Benchmark 4 ratings to each image's magnification. It needs
`judge_output/evaluator_*/pathopen_image_augmentation_eval_data.csv` from
`data_evaluation/vlm/`, plus the pooled pathologist ratings.

In [ ]:
import glob
import re

VLM_DIR = os.path.join(REPO_ROOT, "data_evaluation", "vlm")
HUMAN_INPUT_DIR = os.path.join(REPO_ROOT, "data_evaluation", "pathologists",
                               "scoring_analysis", "input")

# (human_column, judge_column, criterion) - same Benchmark 4 map as
# judge_pathologist_agreement.ipynb; see that notebook for how the positions were verified.
B4_COLUMNS = [
    ("Evaluation Image_Augmentation\n(Benchmark 4)",
     "Evaluation Image_Augmentation\n(Benchmark 4)", "Clinical Relevance"),
    ("Unnamed: 6", "OE_Image_Augmentation_1_VisGround", "Visual Grounding"),
    ("Evaluation Image_Augmentation\n(Benchmark 4).1",
     "Evaluation Image_Augmentation\n(Benchmark 4).1", "Clinical Relevance"),
    ("Unnamed: 10", "OE_Image_Augmentation_2_VisGround", "Visual Grounding"),
]

magnification_by_image = dict(zip(images_df.image_id, images_df.Magnification))

# The Benchmark 4 CSVs key on the AUGMENTED filename ("img_pathopen_148_01_aug_0.png"),
# whereas magnification is recorded against the parent image ("img_pathopen_148_01").
# Strip the "_aug_N.<ext>" suffix to get back to the parent. Without this every row falls
# through to "(unknown)" and the whole stratification silently collapses into one bin.
_AUG_SUFFIX = re.compile(r"_aug_\d+\.(?:png|jpg|jpeg)$", re.IGNORECASE)


def parent_image_id(augmented_filename: str) -> str:
    return _AUG_SUFFIX.sub("", str(augmented_filename).strip())


def _long_scores(df: pd.DataFrame, side: str) -> pd.DataFrame:
    """One row per (image, question-block, criterion) rating, tagged with magnification."""
    idx = 0 if side == "human" else 1
    out = []
    for human_col, judge_col, criterion in B4_COLUMNS:
        col = (human_col, judge_col)[idx]
        if col not in df.columns:
            continue
        scores = pd.to_numeric(df[col], errors="coerce")
        for augmented_id, score in zip(df["Image_ID"], scores):
            if pd.isna(score):
                continue
            parent = parent_image_id(augmented_id)
            out.append({"augmented_id": augmented_id, "image_id": parent,
                        "criterion": criterion, "score": int(score),
                        "magnification": magnification_by_image.get(parent, "(unknown)")})
    return pd.DataFrame(out)


frames = []
for evaluator_dir in sorted(glob.glob(os.path.join(HUMAN_INPUT_DIR, "evaluator[0-9]*")),
                            key=lambda p: int(os.path.basename(p).replace("evaluator", ""))):
    path = os.path.join(evaluator_dir, "pathopen_image_augmentation_eval_data.csv")
    if os.path.exists(path):
        frames.append(pd.read_csv(path).drop(index=0).reset_index(drop=True))
human_aug = pd.concat(frames, ignore_index=True)

long_frames = [_long_scores(human_aug, "human").assign(rater="human")]
for model_key in ["internvl", "qwenvl"]:
    judge_path = os.path.join(VLM_DIR, "judge_output", f"evaluator_{model_key}",
                              "pathopen_image_augmentation_eval_data.csv")
    if os.path.exists(judge_path):
        long_frames.append(_long_scores(pd.read_csv(judge_path), "judge").assign(rater=model_key))
    else:
        print(f"skipping {model_key}: {judge_path} not found")

b4_long = pd.concat(long_frames, ignore_index=True)

# Guard the join: if the suffix convention ever changes, this fails loudly instead of
# quietly reporting one meaningless "(unknown)" bin.
_unknown = (b4_long.magnification == "(unknown)").sum()
print(f"{len(b4_long)} Benchmark 4 ratings across {b4_long.rater.nunique()} raters")
print(f"unresolved magnification: {_unknown}/{len(b4_long)} ({100 * _unknown / len(b4_long):.1f}%)")
assert _unknown < 0.5 * len(b4_long), (
    "more than half the Benchmark 4 ratings could not be mapped to a magnification - "
    "the Image_ID -> parent-image convention has changed"
)
print("magnifications present:", sorted(b4_long.magnification.unique()))
b4_long.head()


In [ ]:
MIN_BIN_N = 30  # per the paper's own "flag small per-bin sample sizes" reviewer concern


def _mag_sort_key(m: str) -> float:
    try:
        return float(str(m).lower().replace("x", ""))
    except ValueError:
        return float("inf")  # unknown/unspecified sorts last


mag_rows = []
for (rater, criterion, mag), grp in b4_long.groupby(["rater", "criterion", "magnification"]):
    scores = grp["score"]
    mag_rows.append({
        "rater": rater, "criterion": criterion, "magnification": mag, "n": len(scores),
        "mean": round(scores.mean(), 3),
        "pct_score_2": round(100 * (scores == 2).mean(), 1),
        "pct_neg1": round(100 * (scores == -1).mean(), 1),
        "low_n_flag": len(scores) < MIN_BIN_N,
    })

magnification_quality = pd.DataFrame(mag_rows).sort_values(
    ["rater", "criterion", "magnification"], key=lambda s: s.map(_mag_sort_key) if s.name == "magnification" else s)
print(magnification_quality.to_string(index=False))
print(f"\nBins flagged below n={MIN_BIN_N}: {int(magnification_quality.low_n_flag.sum())} "
      f"of {len(magnification_quality)}")
magnification_quality

In [ ]:
from scipy.stats import kruskal

# Does augmentation quality actually depend on magnification, or is the spread noise?
# Kruskal-Wallis (non-parametric - the scores are ordinal, not interval), restricted to
# bins that clear MIN_BIN_N so a 2-item bin cannot drive the result.
trend_rows = []
for (rater, criterion), grp in b4_long.groupby(["rater", "criterion"]):
    groups = [g["score"].values for _, g in grp.groupby("magnification") if len(g) >= MIN_BIN_N]
    if len(groups) < 2:
        continue
    stat, p = kruskal(*groups)
    trend_rows.append({"rater": rater, "criterion": criterion, "n_bins_tested": len(groups),
                       "kruskal_H": round(stat, 3), "p_value": f"{p:.4g}",
                       "significant_0.05": p < 0.05})

magnification_trend = pd.DataFrame(trend_rows)
print(magnification_trend.to_string(index=False))
magnification_trend

## The selection manifest: which augmentation files the variant actually contains

Everything above is aggregate - entropy, Gini, per-class counts. None of it names the
individual augmentations the balanced variant keeps, so on its own it is a statistic
rather than a usable dataset.

This section emits the manifest: **one row per retained augmentation file**, with its
path, parent image, case, and the metadata that drove its selection.

**Written to `pathopen_data/augmented/balanced_manifests/`**, not to this notebook's
analysis-output folder. A manifest is a *dataset artifact* - it defines which files a
training run consumes - so it belongs with everything else PathOPEN ships, rather than
somewhere a consumer would need to know the analysis layout to find. The entropy/Gini
tables stay in `augmentation_balance_output/`, since those are results *about* the data.

**Manifest, not a folder of copies.** The augmentation files already exist under
`pathopen_data/processed/images/{image_id}/`. Copying ~850 of them into a new directory
per balanced field would multiply the on-disk footprint, and a training pipeline can read
a path list just as easily. It also keeps the variant cheap to regenerate when
`MAX_MULTIPLIER` changes - rewriting a CSV rather than re-copying image files. Paths in
`relative_path` are repo-relative, so they resolve the same way from any working directory.

**Selection is seeded.** Each image's 30 augmentations sit in arbitrary order in
`data.json`, so "keep k of 30" needs a rule. Taking the first k would bias toward
whatever order the pipeline happened to write, so the choice is a seeded shuffle:
reproducible across runs and machines, but not systematically tied to filename order.

One manifest is written **per metadata field**, because balancing is per-field (see the
Limitations section) - the Organ-balanced variant and the Categorization-balanced
variant are different datasets and must not be conflated.

In [ ]:
SELECTION_SEED = 20260817  # fixed so the manifest is byte-identical across runs

# Augmentation filenames live in data.json; index them by parent image once.
augmented_files_by_image, image_dir_by_image = {}, {}
for case in cases:
    for image in case["Images"]:
        directory = image.get("Directory", "").strip()
        if not directory:
            continue
        image_id = os.path.basename(directory)
        augmented_files_by_image[image_id] = list(image.get("Augmented", []))
        # Repo-relative, so a manifest row resolves the same way regardless of which
        # directory a training script runs from.
        image_dir_by_image[image_id] = os.path.join(
            "pathopen_data", "processed", "images", image_id)


def build_manifest(df: pd.DataFrame, field: str, max_multiplier: int) -> pd.DataFrame:
    """One row per RETAINED augmentation file for the field-balanced variant.

    Which k of an image's 30 augmentations are kept is decided by a seeded shuffle: the
    order in data.json is arbitrary, so taking the first k would bake in whatever order
    the augmentation pipeline happened to write. Seeding keeps the manifest reproducible
    while avoiding that bias."""
    multipliers = targeted_multipliers(df, field, max_multiplier)
    rng = np.random.default_rng(SELECTION_SEED)
    rows = []
    for (_, image), keep in zip(df.iterrows(), multipliers):
        image_id = image["image_id"]
        available = augmented_files_by_image.get(image_id, [])
        if not available:
            continue
        keep = int(min(keep, len(available)))
        # Shuffle a copy - never mutate the source list, which other cells still read.
        chosen = list(rng.permutation(available))[:keep]
        for rank, filename in enumerate(sorted(chosen), start=1):
            rows.append({
                "augmented_filename": filename,
                "relative_path": os.path.join(image_dir_by_image[image_id], filename),
                "image_id": image_id,
                "case_id": image["case_id"],
                "balanced_on": field,
                "class": image[field] if image[field] else "(unspecified)",
                "magnification": image["Magnification"] or "(unknown)",
                "retained_of_30": keep,
                "selection_rank": rank,
            })
    return pd.DataFrame(rows)


manifests = {}
for field in METADATA_FIELDS:
    manifest = build_manifest(images_df, field, MAX_MULTIPLIER)
    manifests[field] = manifest

    # The manifest must reproduce the aggregate counts computed earlier, or the two
    # halves of this notebook disagree about what the variant is.
    from_manifest = collections.Counter(manifest["class"])
    expected = targeted_class_counts(
        images_df, field, targeted_multipliers(images_df, field, MAX_MULTIPLIER))
    assert from_manifest == expected, (
        f"{field}: manifest class counts do not match the reported distribution")

    slug = field.lower().replace(" ", "_")
    path = os.path.join(MANIFEST_DIR, f"balanced_by_{slug}.csv")
    manifest.to_csv(path, index=False)
    print(f"{os.path.relpath(path, REPO_ROOT)}\n    {len(manifest)} files, "
          f"{manifest.image_id.nunique()} parent images, {manifest['class'].nunique()} classes")

# Spot-check that the referenced files actually exist on disk, rather than trusting
# data.json's listing. Sampled, not exhaustive - a full stat() of ~3200 paths is slow.
sample = manifests[METADATA_FIELDS[0]].sample(min(40, len(manifests[METADATA_FIELDS[0]])),
                                              random_state=0)
missing = [p for p in sample.relative_path if not os.path.exists(os.path.join(REPO_ROOT, p))]
print(f"\nspot-check: {len(sample) - len(missing)}/{len(sample)} sampled files present on disk")
if missing:
    print("  MISSING:", missing[:5])

manifests[METADATA_FIELDS[0]].head()


## Save outputs

In [ ]:
outputs = {
    "pillar5b_balance_before_after.csv": balance_df,
    "pillar5b_multiplier_sweep.csv": sweep_df,
    "figure10b_magnification_quality.csv": magnification_quality,
    "figure10b_magnification_trend.csv": magnification_trend,
}
for filename, frame in outputs.items():
    path = os.path.join(OUTPUT_DIR, filename)
    frame.to_csv(path, index=False)
    print(f"{path}  {frame.shape}")

# Per-class counts for the Figure 11 bar charts themselves (Panel A raw, Panel B targeted).
class_rows = []
for field in METADATA_FIELDS:
    for label, count in raw_counts[field].items():
        class_rows.append({"field": field, "class": label, "stage": "raw", "count": count})
    for label, count in targeted_counts[field].items():
        class_rows.append({"field": field, "class": label,
                           "stage": f"targeted_max{MAX_MULTIPLIER}x", "count": count})
class_counts_df = pd.DataFrame(class_rows)
class_path = os.path.join(OUTPUT_DIR, "pillar5b_class_counts.csv")
class_counts_df.to_csv(class_path, index=False)
print(f"{class_path}  {class_counts_df.shape}")

## Limitations

- **The targeted variant is a retention plan, not new imagery.** It selects among
  augmentations that already exist, so it adds no clinical information - §2.5's stated
  reviewer concern ("augmented dataset size must not be conflated with new clinical
  information") applies with full force, and the raw 157 curated cases remain the only
  evaluation-grade benchmark.
- **Balancing is per-field.** Organ, categorization, and regional anatomy are not
  independent (regional anatomy nests inside organ), so balancing one perturbs the
  others. Each field's numbers describe balancing *that* field alone; there is no single
  variant that flattens all of them at once, and claiming one would be unsound.
  Magnification is the partial exception - it is an image-level property rather than a
  case-level one, so a magnification-balanced variant does not redistribute organs or
  categories the way the other three redistribute each other.
- **Magnification balancing is not in the paper's stated scope.** §3.5 names organ,
  categorization, and regional anatomy only. The magnification variant is reported as an
  addition, justified by the 400x quality drop measured below; if §2.5 is not updated to
  claim it, it belongs in supplementary material rather than Figure 11.
- **The multiplier cap bounds achievable balance.** With `MAX_MULTIPLIER` below the
  raw imbalance ratio, perfect balance is unreachable by construction. The sweep above
  quantifies exactly how much correction each cap delivers.
- **Magnification is a proxy for transform sensitivity, not a measurement of it.** It
  answers "does augmentation degrade fine-detail images more?" but cannot attribute any
  effect to a specific transform, because the sampled transforms were never recorded.
  Recovering the original Panel B would require the pipeline to emit a per-image
  transform manifest on a future augmentation run.
- **Two images excluded** (cases 9 and 79) for having no augmentations and an empty
  directory entry.